# SP519 · Prédiction du flux d'électrons de 1 MeV en LEO (NOAA-15)

## Step 1 : préparation des données et analyse statistique
Le notebook suit, dans l'ordre, les 7 points du Step 1 de l'énoncé :

1. Récupérer les données NOAA-15 SEM-2 et les paramètres solaires pour 2020–2024
2. Nettoyer et synchroniser les données (valeurs manquantes, valeurs invalides, outliers, horodatage)
3. Choisir la région des ceintures de radiation (plage de L) et justifier ce choix
4. Moyenner tous les paramètres sur des intervalles de 12 h
5. Visualiser les séries temporelles (flux vs temps, L vs temps, paramètres solaires vs temps)
6. Statistiques descriptives, relations entre le flux et les paramètres solaires, outliers
7. Déterminer le seuil du 95ᵉ percentile du flux, qui définit les « événements »

(La cellule « 0 » ci-dessous contient seulement les imports et les paramètres.)

## 0. Imports et paramètres

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'lines.linewidth': 1.0, 'font.size': 9})

# Dossiers (relatifs au notebook)
PROJ = os.getcwd()
DATA = os.path.join(PROJ, 'data')
FIG = os.path.join(PROJ, 'figures')
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# Paramètres de l'étude
START, END = pd.Timestamp('2020-01-01'), pd.Timestamp('2025-01-01')   # période [START, END[
STEP = '12h'            # résolution temporelle des prédictions
# La région de la ceinture (L_LO <= L < L_HI) est choisie automatiquement au point 3
P_EVENT = 95            # percentile définissant un "événement"

def savefig(fig, name):
    # Sauvegarde des figures pour le rapport
    fig.savefig(os.path.join(FIG, name), dpi=150, bbox_inches='tight')

print('Dossier du projet :', PROJ)

## 1. Récupération des données (NOAA-15 SEM-2 + paramètres solaires, 2020–2024)

### 1.1 Téléchargement des fichiers bruts NOAA-15
Un fichier par jour sur le serveur NCEI (`poes_n15_AAAAMMJJ_proc.nc`, ~6,6 Mo, une mesure toutes les 2 s), soit
~1 800 fichiers et ~12 Go pour 2020–2024. Ils sont rangés dans `data/noaa15_raw/` (ignoré par Git).
Le téléchargement est **désactivé par défaut** (`TELECHARGER = False`) pour que le notebook puisse s'exécuter de haut
en bas sans retélécharger 12 Go : les tables résumées du point 1.2 sont déjà dans `data/`.

In [ ]:
import calendar
from concurrent.futures import ThreadPoolExecutor
from typing import List

import requests

TELECHARGER = False                          # True pour (re)télécharger les fichiers bruts NOAA-15
RAW_DIR = os.path.join(DATA, 'noaa15_raw')   # fichiers bruts (ignorés par Git)
MAX_WORKERS = 4                              # au-delà, le serveur NCEI bloque des connexions

def download_nc_files(
    years: List[int],
    months: List[int],
    data_dir: str = RAW_DIR,
    max_workers: int = MAX_WORKERS,
) -> None:
    """Downloads NOAA-15 POES daily NetCDF files into data_dir.

    Files already present are skipped (the download can be resumed), and a file
    whose download fails is deleted (no half-written file is left behind).

    Args:
        years: List of years (e.g., [2020, 2021]).
        months: List of months as integers; downloads every day in each month.
        data_dir: Path to the local directory where files will be stored.
        max_workers: Maximum number of files to download concurrently.
    """
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1")

    files_to_download = []
    for year in years:
        for month in months:
            if not 1 <= month <= 12:
                raise ValueError(f"Month must be between 1 and 12, got {month}")
            for day in range(1, calendar.monthrange(year, month)[1] + 1):
                date_str = f"{year:04d}{month:02d}{day:02d}"
                files_to_download.append(
                    (
                        year,
                        date_str,
                        f"poes_n15_{date_str}_proc.nc",
                    )
                )

    # 1. Create the destination directory (existing files are kept)
    os.makedirs(data_dir, exist_ok=True)

    base_url = (
        "https://www.ncei.noaa.gov/data/poes-metop-space-environment-monitor/"
        "access/l1b/v01r00/{year}/noaa15/poes_n15_{date_str}_proc.nc"
    )

    def download_file(file_info):
        year, date_str, filename = file_info
        file_url = base_url.format(year=year, date_str=date_str)
        destination_path = os.path.join(data_dir, filename)
        if os.path.exists(destination_path):
            return                                   # already downloaded
        tmp_path = destination_path + ".part"        # renamed only once complete

        print(f"Downloading: {file_url}")
        try:
            with requests.get(file_url, stream=True, timeout=30) as response:
                response.raise_for_status()

                with open(tmp_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

            os.replace(tmp_path, destination_path)
            print(f"Successfully saved to {destination_path}")

        except requests.exceptions.RequestException as e:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            print(f"Failed to download {file_url}: {e}")

    # 2. Download daily files concurrently to reduce time spent waiting on the server.
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(executor.map(download_file, files_to_download))


if TELECHARGER:
    download_nc_files(years=list(range(START.year, END.year)), months=list(range(1, 13)))
else:
    print('Téléchargement désactivé (TELECHARGER = False) : on utilise les tables déjà présentes dans data/.')

### 1.2 Paramètres solaires et géomagnétiques (OMNI)
Base **OMNI2 horaire** (`OMNI2_H0_MRG1HR`) de la NASA (SPDF / CDAWeb) : mesures du vent solaire (sondes ACE, Wind…)
ramenées jusqu'à la Terre, et indices géomagnétiques, sur une même base de temps horaire UTC. Deux fichiers CDF par an
(janvier–juin, juillet–décembre), soit 10 fichiers d'environ 1,5 Mo, lus avec `cdflib`.

| Variable OMNI | Nom dans le notebook | Rôle physique |
|---|---|---|
| `V` | `V_kms` | vitesse du vent solaire : principal moteur de l'accélération des électrons relativistes |
| `BZ_GSM` | `Bz_nT` | Bz < 0 → reconnexion magnétique, injection de particules dans la magnétosphère |
| `F10_INDEX` | `F107_sfu` | flux radio solaire à 10,7 cm : niveau d'activité du cycle solaire |
| `N`, `Pressure` | `N_cm3`, `P_nPa` | densité et pression dynamique : compression de la magnétosphère, pertes d'électrons |
| `E` | `E_mVm` | champ électrique −V·Bz : couplage vent solaire / magnétosphère |
| `ABS_B` | `B_nT` | intensité du champ magnétique interplanétaire |
| `KP`, `DST`, `AE` | `Kp`, `Dst_nT`, `AE_nT` | activité géomagnétique : globale (Kp), orages (Dst), sous-orages (AE) |

V, Bz et F10.7 sont demandés par l'énoncé ; les autres sont ajoutés pour les raisons physiques ci-dessus.
Les valeurs manquantes sont codées dans les CDF par une valeur de remplissage (attribut `FILLVAL`, ex. 9999 pour V) :
on les remplace par NaN. `KP` est stocké ×10 (Kp = 3+ → 33) : on le divise par 10.
Le flux de protons > 10 MeV (`PR-FLX_10`) n'est pas gardé : il manque à ~97 % sur 2020–2024.

In [ ]:
import cdflib

OMNI_DIR = os.path.join(DATA, 'omni')   # fichiers CDF bruts (ignorés par Git)
OMNI_URL = 'https://spdf.gsfc.nasa.gov/pub/data/omni/omni_cdaweb/hourly/{y}/omni2_h0_mrg1hr_{y}{m}01_v01.cdf'
OMNI_VARS = {'V': 'V_kms', 'BZ_GSM': 'Bz_nT', 'F10_INDEX': 'F107_sfu', 'N': 'N_cm3', 'Pressure': 'P_nPa',
             'E': 'E_mVm', 'ABS_B': 'B_nT', 'KP': 'Kp', 'DST': 'Dst_nT', 'AE': 'AE_nT'}
os.makedirs(OMNI_DIR, exist_ok=True)

morceaux = []
for y in range(START.year, END.year):
    for m in ('01', '07'):                                   # un fichier par semestre
        path = os.path.join(OMNI_DIR, f'omni2_h0_mrg1hr_{y}{m}01_v01.cdf')
        if not os.path.exists(path):                         # téléchargé une seule fois
            r = requests.get(OMNI_URL.format(y=y, m=m), timeout=60)
            r.raise_for_status()
            with open(path, 'wb') as f:
                f.write(r.content)
        cdf = cdflib.CDF(path)
        d = {'time': cdflib.cdfepoch.to_datetime(cdf.varget('Epoch'))}
        for var, nom in OMNI_VARS.items():
            x = cdf.varget(var).astype(float)
            fill = cdf.varattsget(var)['FILLVAL']
            x[np.isclose(x, fill) | (x >= fill)] = np.nan       # valeur de remplissage -> NaN
            d[nom] = x
        morceaux.append(pd.DataFrame(d))

omni_raw = pd.concat(morceaux).drop_duplicates('time').set_index('time').sort_index()
omni_raw = omni_raw.loc[START:END - pd.Timedelta('1h')]
omni_raw['Kp'] = omni_raw['Kp'] / 10                         # Kp*10 -> Kp
omni_raw.index = pd.DatetimeIndex(omni_raw.index).as_unit('ns')
omni_raw.to_parquet(os.path.join(DATA, 'omni_hourly_2020_2024.parquet'))
print('OMNI :', omni_raw.shape, '|', omni_raw.index.min(), '->', omni_raw.index.max())

### 1.3 Tables utilisées dans la suite du notebook
La cellule ci-dessous lit les deux tables de `data/` ; tout le reste du notebook ne dépend que d'elles.

* `noaa15_p6_hourly_Lbins.parquet` : le flux `mep_pro_tel90_flux_p6` de NOAA-15, résumé **heure par heure et par
  tranche de L** (L de 1 à 8 par pas de 0,25, et `Lbin = 10` pour la calotte polaire L ≥ 10).
  Colonnes : `time` (début de l'heure, UTC), `Lbin`, `p6_sum` (somme des flux), `n` (nombre de mesures),
  `p6_max` (plus grande mesure de 2 s de la case, utile pour repérer les pics aberrants),
  `omni_p1_sum` / `omni_p1_n` (idem pour les protons 25 MeV, utiles au point 2).
  → Produite au **point 2** à partir des fichiers bruts (nettoyage puis résumé) : **en cours, autre membre du
  binôme**. En attendant, la version présente dans `data/` a été produite selon le même principe.
* `omni_hourly_2020_2024.parquet` : paramètres OMNI heure par heure (index `time`, UTC) : `V_kms`, `Bz_nT`,
  `F107_sfu`, `N_cm3`, `P_nPa`, `E_mVm`, `B_nT`, `Kp`, `Dst_nT`, `AE_nT`.
  → Produite par la cellule OMNI ci-dessus (1.2).

In [ ]:
noaa = pd.read_parquet(os.path.join(DATA, 'noaa15_p6_hourly_Lbins.parquet'))
omni = pd.read_parquet(os.path.join(DATA, 'omni_hourly_2020_2024.parquet'))

L_POLAR = 10.0   # code de la tranche "calotte polaire" (L >= 10)

print('NOAA-15 :', noaa.shape, '|', noaa.time.min(), '->', noaa.time.max())
print('   tranches de L :', [float(l) for l in sorted(noaa.Lbin.unique())])
jours = noaa.time.dt.floor('D').unique()
print('   jours avec données :', len(jours), '/', (END - START).days)
print('OMNI    :', omni.shape, '|', omni.index.min(), '->', omni.index.max())
print('   % manquant :', omni.isna().mean().mul(100).round(2).to_dict())
noaa.head()

## 2. Nettoyage et synchronisation

Code écrit par l'autre membre du binôme (`cleaner.py`), intégré ici. Il travaille sur les **mesures brutes** (une
toutes les 2 s, un fichier par jour) et fait uniquement ce que demande le point 2 de l'énoncé :

* **horodatage** (`as_dataframe`) : temps converti en dates UTC, triées, sans doublon ;
* **valeurs invalides / manquantes** (`clean_raw`) : flux négatifs (valeurs de remplissage −999) → NaN ; L invalide
  (−1 = valeur de remplissage, L ≈ 100 = ligne de champ ouverte) ; mesures prises pendant l'étalonnage en vol
  (`mep_IFC_on = 1`) retirées ; mesures sans flux ou sans L retirées ;
* **valeurs aberrantes** (`mask_outliers`) : dans chaque tranche de L de largeur 1, une mesure est aberrante si son
  log10(flux) sort de [Q1 − 3·IQR ; Q3 + 3·IQR] (IQR = Q3 − Q1). Les flux nuls sont de vraies mesures (aucun coup
  compté) : ils sont gardés ;
* **synchronisation** (`synchronize`) : met d'autres séries (OMNI : V, Bz, F10.7…) sur les dates NOAA-15 (valeur la
  plus proche, à moins d'1 h).

### 2.1 Fonctions de nettoyage

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd

TARGET = "mep_pro_tel90_flux_p6"
L_COL = "L_IGRF"
P6_COUNT_STEP = 243.9024  # flux of one count in the P6 channel [#/cm2-s-str]
L_VALID_RANGE = (1.0, 20.0)  # L outside is a fill value (-1) or an open field line

RAW_COLUMNS = (TARGET, "mep_pro_tel0_flux_p6", "mep_ele_tel90_flux_e3", L_COL, "MLT",
               "mag_lat_sat", "lat", "lon", "alt", "Btot_sat", "mep_IFC_on")


@dataclass
class CleanConfig:
    target: str = TARGET
    l_col: str = L_COL
    drop_ifc: bool = True  # drop samples taken during in-flight calibration (flag == 1)
    outlier_k: float | None = 3.0  # IQR multiplier on log10 flux (None = keep outliers)
    outlier_l_bin: float = 1.0  # outliers are searched inside L bins of this width
    outlier_min_group: int = 50  # minimum positive samples in a bin to apply the rule


# --------------------------------------------------------------------------------------
# Time stamps
# --------------------------------------------------------------------------------------
def _to_datetime_index(t) -> pd.DatetimeIndex:
    t = pd.Series(np.asarray(t))
    if pd.api.types.is_datetime64_any_dtype(t):
        return pd.DatetimeIndex(t)
    v = t.astype("int64")  # NOAA 'time' is uint64 milliseconds since 1970
    m = v.abs().max()
    unit = "ns" if m > 1e17 else "us" if m > 1e14 else "ms" if m > 1e11 else "s"
    return pd.DatetimeIndex(pd.to_datetime(v, unit=unit))


def as_dataframe(data, columns: Iterable[str] | None = None) -> pd.DataFrame:
    """Loader chunk -> float64 DataFrame indexed by naive-UTC, sorted, unique time stamps.

    Accepts a DataFrame, an xarray.Dataset, a dict of arrays or a NetCDF path. For xarray the
    variables are read one by one (``columns`` defaults to ``RAW_COLUMNS``): in the NCEI files
    every variable is also its own dimension, so ``Dataset.to_dataframe()`` would explode.
    """
    if isinstance(data, (str, os.PathLike)):
        import xarray as xr

        data = xr.open_dataset(data, decode_times=False)

    if isinstance(data, pd.DataFrame):
        df = data.copy()
        if columns is not None:
            keep = [c for c in columns if c in df.columns]
            df = df[keep + (["time"] if "time" in df.columns and "time" not in keep else [])]
    elif hasattr(data, "variables") and hasattr(data, "attrs"):  # xarray.Dataset
        names = [c for c in (columns or RAW_COLUMNS) if c in data.variables]
        df = pd.DataFrame({n: np.asarray(data[n].values) for n in names})
        if "time" in data.variables:
            df["time"] = np.asarray(data["time"].values)
    else:
        df = pd.DataFrame({k: np.asarray(v) for k, v in dict(data).items()})
        if columns is not None:
            df = df[[c for c in [*columns, "time"] if c in df.columns]]

    if isinstance(df.index, pd.DatetimeIndex):
        idx = df.index
    elif "time" in df.columns:
        idx = _to_datetime_index(df.pop("time").to_numpy())
    else:
        raise KeyError("no DatetimeIndex and no 'time' column in the data")
    if idx.tz is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)
    df.index = pd.DatetimeIndex(idx, name="time")
    return df.astype("float64")


# --------------------------------------------------------------------------------------
# Invalid / missing values and outliers
# --------------------------------------------------------------------------------------
def mask_outliers(df: pd.DataFrame, cfg: CleanConfig) -> pd.Series:
    """Boolean mask of outliers of the target: IQR rule on log10 flux, per L bin.

    Zero-count samples are real values and are neither used for the quartiles nor flagged.
    Bins with fewer than ``outlier_min_group`` positive samples are left untouched.
    """
    out = pd.Series(False, index=df.index)
    if cfg.outlier_k is None:
        return out
    flux = df[cfg.target]
    logv = np.log10(flux.where(flux > 0))
    lbin = np.floor(df[cfg.l_col] / cfg.outlier_l_bin)
    g = logv.groupby(lbin)
    q = g.quantile([0.25, 0.75]).unstack()
    n = g.count()
    ok = n >= cfg.outlier_min_group
    q1, q3 = lbin.map(q[0.25].where(ok)), lbin.map(q[0.75].where(ok))
    iqr = q3 - q1
    return ((logv < q1 - cfg.outlier_k * iqr) | (logv > q3 + cfg.outlier_k * iqr)).fillna(False)


def clean_raw(df: pd.DataFrame, cfg: CleanConfig | None = None) -> tuple[pd.DataFrame, dict]:
    """Remove invalid and missing values, flag outliers. Returns ``(clean_df, report)``."""
    cfg = cfg or CleanConfig()
    for c in (cfg.target, cfg.l_col):
        if c not in df.columns:
            raise KeyError(f"column {c!r} missing (got {list(df.columns)})")
    df = df.copy()
    rep = {"n_raw": len(df),
           "duplicate_timestamps": int(df.index.duplicated().sum()),
           "unsorted_timestamps": int((df.index.to_series().diff() < pd.Timedelta(0)).sum())}
    df = df[~df.index.duplicated(keep="first")].sort_index()

    # fill values (-999 in several MEPED channels) are negative fluxes -> NaN
    for c in [c for c in df.columns if "flux" in c]:
        bad = df[c] < 0
        if bad.any():
            rep[f"fill_values_{c}"] = int(bad.sum())
            df.loc[bad, c] = np.nan

    # invalid L: -1 fill value, open field lines (L ~ 100)
    bad_l = ~df[cfg.l_col].between(*L_VALID_RANGE)
    rep["invalid_L"] = int(bad_l.sum())
    df.loc[bad_l, cfg.l_col] = np.nan

    # in-flight calibration (flag 1 = on; -1 is "not applicable" and kept)
    if cfg.drop_ifc and "mep_IFC_on" in df.columns:
        ifc = df["mep_IFC_on"] == 1
        rep["ifc_on"] = int(ifc.sum())
        df = df[~ifc]

    # missing values: a sample is useless without target or L
    miss = df[[cfg.target, cfg.l_col]].isna().any(axis=1)
    rep["rows_removed_missing"] = int(miss.sum())
    df = df[~miss]

    # outliers -> NaN on the target, then drop those rows
    out = mask_outliers(df, cfg)
    rep["outliers"] = int(out.sum())
    df = df[~out]
    rep["n_clean"] = len(df)
    return df, rep


def clean(data, cfg: CleanConfig | None = None, columns=None) -> tuple[pd.DataFrame, dict]:
    """``as_dataframe`` + ``clean_raw``."""
    return clean_raw(as_dataframe(data, columns), cfg)


# --------------------------------------------------------------------------------------
# Synchronisation
# --------------------------------------------------------------------------------------
def synchronize(base: pd.DataFrame, *others: pd.DataFrame, tolerance: str = "1h") -> pd.DataFrame:
    """Attach other series (e.g. OMNI V, Bz, F10.7) to the NOAA time stamps.

    Each value is the nearest one in time, accepted only within ``tolerance`` (else NaN).
    ``others`` go through ``as_dataframe`` so they can come in any format / time unit.
    """
    out = base.copy()
    tol = pd.Timedelta(tolerance)
    for o in others:
        o = as_dataframe(o)
        o = o[~o.index.duplicated(keep="first")].sort_index()
        o = o.drop(columns=[c for c in o.columns if c in out.columns])
        out = out.join(o.reindex(out.index, method="nearest", tolerance=tol))
    return out


# --------------------------------------------------------------------------------------
# Saving
# --------------------------------------------------------------------------------------
def save_cleaned(df: pd.DataFrame, outdir: str = "data/cleaned", name: str | None = None) -> str | None:
    """Save the cleaned data in ``outdir`` (created if needed) ONLY if the folder is empty.

    Parquet if pyarrow/fastparquet is installed, otherwise gzipped CSV. Returns the written path,
    or None when the folder already contains files (nothing is overwritten).
    """
    os.makedirs(outdir, exist_ok=True)
    if os.listdir(outdir):
        print(f"{outdir} is not empty: nothing saved")
        return None
    name = name or f"cleaned_{df.index[0]:%Y%m%d}_{df.index[-1]:%Y%m%d}"
    try:
        path = os.path.join(outdir, name + ".parquet")
        df.to_parquet(path)
    except ImportError:
        path = os.path.join(outdir, name + ".csv.gz")
        df.to_csv(path)
    print(f"saved {len(df)} rows to {path}")
    return path

### 2.2 Test sur une journée brute
On applique le nettoyage au fichier brut du 01/01/2020 (`data/noaa15_raw/`). Le fichier est lu avec `h5netcdf` :
netCDF4 est bloqué par Windows et ne sait pas ouvrir un chemin contenant un accent (« Donnée »).

In [ ]:
import xarray as xr

def lire_jour_noaa(path, columns=RAW_COLUMNS):
    """Lit un fichier brut NOAA-15 (1 jour, une mesure toutes les 2 s).
    decode_times=False : on garde les nombres bruts, as_dataframe convertit le temps lui-même."""
    with xr.open_dataset(path, engine='h5netcdf', decode_times=False, decode_timedelta=False) as ds:
        return as_dataframe(ds, columns)

brut = lire_jour_noaa(os.path.join(RAW_DIR, 'poes_n15_20200101_proc.nc'))
propre, rapport = clean_raw(brut, CleanConfig())

print(f'{len(brut)} mesures brutes -> {len(propre)} mesures propres')
for k, v in rapport.items():
    print(f'  {k:<25}{v:>8}')

## 3. Choix de la région des ceintures de radiation (plage de L)

**Le paramètre L** : distance (en rayons terrestres) à laquelle la ligne de champ magnétique sur laquelle se trouve
le satellite croise l'équateur magnétique. NOAA-15 est en orbite basse (~810 km) mais, en passant d'un pôle à
l'autre, il traverse toutes les lignes de champ, donc toutes les valeurs de L. Le flux mesuré dépend énormément de L :
il faut choisir **quelle région** on cherche à prédire.

**Ce que dit la physique :**
* **L < 2 : ceinture interne.** Peuplée de protons piégés très énergétiques (Anomalie Sud-Atlantique) : P6 y compte
  surtout des **protons**, pas des électrons → à exclure.
* **L ≈ 2–3 : zone « slot ».** Quasi vide d'électrons, sauf après de très forts orages.
* **L ≈ 3–7 : ceinture externe.** C'est là que se trouvent les électrons relativistes (~1 MeV) qui varient fortement
  avec le vent solaire : c'est la région qui nous intéresse.
* **L > 7 :** on sort des ceintures (lignes de champ ouvertes ou très étirées) → flux au niveau du bruit.

**Critères de choix**, vérifiés sur les données 2020–2024 ci-dessous. La région retenue doit :
1. avoir un flux **nettement au-dessus du bruit de fond** de l'instrument (sinon on prédit du bruit) ;
2. être **variable** dans le temps (sinon il n'y a rien à prédire) ;
3. être **liée au vent solaire** (sinon les paramètres OMNI ne serviront à rien pour prédire).

### 3.1 Carte L – temps du flux
Pour chaque jour et chaque tranche de L, flux moyen = Σ flux / Σ mesures. L'échelle de couleur est logarithmique
(le flux varie sur plusieurs ordres de grandeur). L'indice Dst est tracé en dessous : les fortes valeurs négatives
signalent les orages géomagnétiques.

In [ ]:
# Niveau de bruit : dans la calotte polaire (L >= 10) il n'y a pas d'électrons piégés,
# P6 n'y mesure que le bruit de fond de l'instrument
pc = noaa[noaa.Lbin == L_POLAR]
NOISE = (pc.p6_sum / pc.n).median()
print(f'Bruit de fond P6 (médiane horaire en calotte polaire) : {NOISE:.0f} #/cm²/s/sr')

# Moyenne journalière par tranche de L (L < 8)
belt = noaa[noaa.Lbin < L_POLAR]
d = belt.groupby([belt.time.dt.floor('D'), 'Lbin'])[['p6_sum', 'n']].sum()
carte = (d.p6_sum / d.n).unstack('Lbin')
carte = carte.reindex(pd.date_range(START, END, freq='D', inclusive='left'))   # jours manquants -> NaN

fig, ax = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
cmap = plt.get_cmap('viridis').copy()
cmap.set_bad('0.85')                                      # jours manquants en gris
im = ax[0].imshow(np.log10(carte.T.clip(lower=NOISE / 2)).values, aspect='auto', origin='lower', cmap=cmap,
                  extent=[mdates.date2num(START), mdates.date2num(END), 1, 8], interpolation='nearest')
ax[0].grid(False)
ax[0].set_ylabel('L (IGRF)')
ax[0].set_title('NOAA-15 P6 (électrons ~1 MeV) : flux moyen journalier par tranche de L, 2020–2024')
fig.colorbar(im, ax=ax, label='log10 flux P6 (#/cm²/s/sr)', pad=0.01, fraction=0.03)
ax[1].plot(omni.index, omni['Dst_nT'], color='C0', lw=0.5)
ax[1].set_ylabel('Dst (nT)')
ax[1].xaxis_date()
savefig(fig, '3_carte_L_temps.png')
plt.show()

### 3.2 Profil en L : niveau, variabilité et lien avec le vent solaire
On calcule, pour chaque tranche de L, le flux moyen sur des fenêtres de 12 h (la résolution du projet), puis :
* le **flux moyen** (linéaire) → sert au choix automatique de la région ;
* la **médiane** et les percentiles 5 % / 95 % du log10 du flux → niveau par rapport au bruit (critère 1) ;
* le **% de fenêtres où le flux dépasse 2 × le bruit** (critère 1) ;
* l'**écart-type du log10 du flux** → variabilité (critère 2) ;
* la **corrélation de Spearman** entre le log10 du flux et la vitesse du vent solaire V **24 h plus tôt**
  (le vent solaire met 1 à 2 jours à accélérer les électrons) → lien avec le vent solaire (critère 3).

In [ ]:
d12 = belt.groupby([pd.Grouper(key='time', freq=STEP), 'Lbin'])[['p6_sum', 'n']].sum()
f12 = (d12.p6_sum / d12.n).unstack('Lbin')                # flux 12 h : une colonne par tranche de L
logf = np.log10(f12.clip(lower=NOISE / 2))
V12 = omni['V_kms'].resample(STEP).mean()

profil = pd.DataFrame({
    'flux moyen': f12.mean(),
    'log10 P5': logf.quantile(0.05),
    'log10 médiane': logf.median(),
    'log10 P95': logf.quantile(0.95),
    '% > 2×bruit': (f12 > 2 * NOISE).mean() * 100,
    'écart-type log10': logf.std(),
    'Spearman V(t-24h)': logf.apply(lambda c: c.corr(V12.shift(2).reindex(c.index), method='spearman')),
})
profil.index.name = 'L (bord gauche)'

# Choix automatique de la région : largeur à mi-hauteur du profil de flux moyen.
# On part de la tranche où le flux moyen est maximal, puis on ajoute les tranches voisines
# tant que leur flux moyen reste >= SEUIL_REGION x le maximum.
SEUIL_REGION = 0.5
LBIN_W = 0.25                                             # largeur d'une tranche de L
flux_moy = profil['flux moyen']
garde = (flux_moy >= SEUIL_REGION * flux_moy.max()).values
lo = hi = flux_moy.index.get_loc(flux_moy.idxmax())
while lo > 0 and garde[lo - 1]:
    lo -= 1
while hi < len(garde) - 1 and garde[hi + 1]:
    hi += 1
L_LO, L_HI = flux_moy.index[lo], flux_moy.index[hi] + LBIN_W

print(f'Flux moyen maximal : {flux_moy.max():.0f} #/cm²/s/sr pour {flux_moy.idxmax()} <= L < {flux_moy.idxmax() + LBIN_W}')
print(f'Région retenue automatiquement : {L_LO} <= L < {L_HI}')
display(profil.round(2))

x = profil.index + LBIN_W / 2                             # centre des tranches
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
ax[0].plot(x, profil['flux moyen'], color='C0', lw=2)
ax[0].axhline(SEUIL_REGION * flux_moy.max(), color='C1', ls='--', label=f'{SEUIL_REGION:.0%} du maximum')
ax[0].set_ylabel('flux moyen (#/cm²/s/sr)')
ax[0].set_title('0. Flux moyen (choix de la région)')
ax[0].legend(fontsize=7)
ax[1].fill_between(x, profil['log10 P5'], profil['log10 P95'], color='C0', alpha=0.25, label='P5 – P95')
ax[1].plot(x, profil['log10 médiane'], color='C0', lw=2, label='médiane')
ax[1].axhline(np.log10(NOISE), color='C3', ls='--', label='bruit')
ax[1].set_ylabel('log10 flux (12 h)')
ax[1].set_title('1. Niveau')
ax[1].legend(fontsize=7)
ax[2].plot(x, profil['écart-type log10'], color='C0', lw=2)
ax[2].set_ylabel('écart-type de log10 flux')
ax[2].set_title('2. Variabilité')
ax[3].plot(x, profil['Spearman V(t-24h)'], color='C0', lw=2)
ax[3].set_ylabel('ρ Spearman')
ax[3].set_title('3. Lien avec V 24 h avant')
for a in ax:
    a.axvspan(L_LO, L_HI, color='C1', alpha=0.2)
    a.set_xlabel('L')
fig.suptitle(f'Profil en L (fenêtres de 12 h, 2020–2024) — zone orange : région retenue {L_LO} ≤ L < {L_HI}')
fig.tight_layout()
savefig(fig, '3_profil_L.png')
plt.show()

### 3.3 Conclusion : vérification de la région choisie automatiquement

**Règle de choix (codée en 3.2) :** la région est la **largeur à mi-hauteur** du profil de flux moyen. On part de
la tranche de L où le flux moyen est maximal (le cœur de la ceinture externe) et on garde les tranches voisines
tant que leur flux moyen reste au moins égal à la moitié du maximum. Aucune valeur de L n'est rentrée à la main ;
le seul paramètre est `SEUIL_REGION = 0.5` (la « mi-hauteur »).

**Ce que montre la carte L – temps (3.1) :**
* une bande **constante à L ≈ 1,25–1,75** : les protons de la ceinture interne, insensibles aux orages → exclue ;
* une **zone « slot » sombre à L ≈ 2–3**, remplie seulement lors des orages les plus forts (2023, mai 2024 :
  Dst ≈ −400 nT), quand la ceinture externe est poussée vers la Terre ;
* une **ceinture externe à L ≈ 3,5–6** qui s'allume et s'éteint au rythme de l'activité solaire : c'est la cible ;
* au-delà de L ≈ 7, le flux reste au niveau du bruit.

La cellule ci-dessous vérifie que la région retenue remplit les trois critères (flux moyenné sur toute la région,
fenêtres de 12 h) et la compare aux deux régions décalées d'une demi-unité de L.

In [ ]:
def criteres(lo, hi):
    """Les 3 critères du point 3 pour la région lo <= L < hi (flux 12 h moyenné sur toute la région)."""
    g = belt[(belt.Lbin >= lo) & (belt.Lbin < hi)].groupby(pd.Grouper(key='time', freq=STEP))[['p6_sum', 'n']].sum()
    f = (g.p6_sum / g.n).dropna()
    lf = np.log10(f.clip(lower=NOISE / 2))
    return {'flux moyen': f.mean(),
            '% > 2×bruit': (f > 2 * NOISE).mean() * 100,
            'écart-type log10': lf.std(),
            'Spearman V(t-24h)': lf.corr(V12.shift(2).reindex(lf.index), method='spearman')}

regions = {f'{L_LO - 0.5} ≤ L < {L_HI - 0.5}': criteres(L_LO - 0.5, L_HI - 0.5),
           f'{L_LO} ≤ L < {L_HI} (retenue)': criteres(L_LO, L_HI),
           f'{L_LO + 0.5} ≤ L < {L_HI + 0.5}': criteres(L_LO + 0.5, L_HI + 0.5)}
display(pd.DataFrame(regions).T.round(2))

**Lecture (2020–2024) : la règle donne 3,5 ≤ L < 5,25**, centrée sur le maximum du flux moyen (L ≈ 4,25–4,5).
Moyennée sur toute la région, elle remplit les trois critères :
1. flux **au-dessus de 2 × le bruit dans 93 % des fenêtres** de 12 h ;
2. **variabilité** forte (écart-type de log10 flux ≈ 0,56) : il y a quelque chose à prédire ;
3. **lien avec le vent solaire** (Spearman avec V 24 h avant ≈ 0,35), au moins aussi bon que ses voisines.

C'est aussi la région qui a le **flux moyen le plus élevé** des trois. Décalée vers le bas (3–4,75), elle déborde
sur le « slot » et passe plus souvent dans le bruit (89 %) ; décalée vers le haut (4–5,75), le flux moyen et la
variabilité baissent.

Limite : la ceinture se déplace (vers la Terre pendant les forts orages). Une plage fixe ne la suit pas toujours ;
comparer plusieurs plages de L est d'ailleurs proposé en bonus par l'énoncé.

## 4 → 7. Suite du Step 1 (à faire par l'autre membre du binôme)

> **En cours.** Variables déjà disponibles en arrivant ici : `noaa` (table horaire par tranche de L : `time`, `Lbin`,
> `p6_sum`, `n`, `p6_max`), `omni` (table OMNI horaire, index = temps), `NOISE` (bruit de fond P6), `STEP = '12h'`,
> `L_LO`, `L_HI` (région choisie automatiquement au point 3), `P_EVENT = 95`, `savefig(fig, nom)`.

**4. Moyennes sur 12 h** (fenêtres 00:00–12:00 et 12:00–24:00 UTC, de `START` à `END`)
* Flux : ne garder que `L_LO <= Lbin < L_HI`, puis flux 12 h = Σ `p6_sum` / Σ `n` (moyenne du flux **linéaire**),
  puis `log10`. Fenêtre invalide (NaN) si Σ `n` < 25 % de la médiane de Σ `n` (trop peu de passages dans la région).
* OMNI : moyenne de chaque paramètre sur la fenêtre, NaN s'il y a moins de 6 heures valides sur 12.
* **Exporter la table** `data/table_12h.parquet` — c'est l'entrée du Step 2 (« Come with your 12-hour table ready ») :
  index `time` (début de la fenêtre), colonnes `flux`, `log_flux`, `n`, `V_kms`, `Bz_nT`, `F107_sfu`, `N_cm3`,
  `P_nPa`, `E_mVm`, `B_nT`, `Kp`, `Dst_nT`, `AE_nT`. **Garder ces noms** : le Step 2 les utilise.

**5. Visualiser les séries temporelles**
* flux 12 h vs temps (échelle log) ;
* L vs temps (L de NOAA-15 sur quelques orbites + couverture de la région `L_LO`–`L_HI` par fenêtre de 12 h) ;
* paramètres solaires vs temps (V, Bz, F10.7, et Dst / Kp).

**6. Statistiques descriptives, relations, outliers**
* `describe()` de la table 12 h, histogramme de `log_flux` ;
* matrice de corrélation (heatmap) flux ↔ paramètres, puis corrélations **retardées** (paramètre à t − 12 h,
  t − 24 h, … t − 72 h) : c'est ce qui dira quels paramètres sont utiles pour prédire à +12 → +48 h ;
* outliers : valeurs extrêmes de la table 12 h (ex. règle IQR sur `log_flux`), et vérifier s'il s'agit de vrais
  orages (Dst très négatif) ou d'artefacts.

**7. Seuil du 95ᵉ percentile**
* seuil = P95 du flux 12 h ; tracer le flux avec la ligne du seuil ; nombre et dates des « événements ».
* Pour le Step 2, ce seuil sera **recalculé sur la seule période d'entraînement** (sinon le modèle « voit » le futur).

**Points encore ouverts du point 2 (nettoyage)**, à régler avant l'export de la table 12 h :
* la règle IQR de `clean_raw` retire de vraies petites mesures (2–4 coups dans le « slot » et la calotte polaire) ;
  la remplacer par un seuil sur les pics impossibles (mesures de 2 s > 1e7, ex. la valeur 97 904 272 répétée) ;
* exclure les périodes de protons solaires (SEP), où P6 ne mesure plus des électrons ;
* appliquer ce nettoyage à toute la période 2020–2024, puis recalculer la table horaire `noaa` à partir des données
  nettoyées.

# Step 2 : modélisation par apprentissage automatique

## 8. Régression : prédire le flux à +12, +24, +36 et +48 h (énoncé 4.1)

### 8.1 Données d'entrée et découpage entraînement / test
On part de la **table 12 h** produite au point 4 (`data/table_12h.parquet`).

**Ce que le modèle reçoit (les entrées X) :** à un instant t, on ne connaît que le passé. On donne donc au modèle
le log10 du flux et les paramètres solaires **à t, t − 12 h et t − 24 h** (le flux réagit au vent solaire avec 1 à
2 jours de retard, les valeurs passées sont donc utiles).

**Ce qu'il doit prédire (les cibles Y) :** le log10 du flux **12, 24, 36 et 48 h plus tard**. On prédit le log10
et pas le flux lui-même, car le flux varie sur plusieurs ordres de grandeur : sans log, les quelques valeurs
d'orage écraseraient tout le reste.

**Découpage honnête (l'énoncé insiste) :** le test doit jouer le rôle d'un **futur que le modèle n'a jamais vu**.
On coupe donc **dans le temps**, et pas au hasard :
* **entraînement : 2020 → 2023** (le modèle apprend) ;
* **test : 2024** (on mesure ses performances), qui contient en plus le gros orage de mai 2024.

Un découpage au hasard tricherait : deux fenêtres de 12 h voisines se ressemblent beaucoup, le modèle aurait vu
« la veille » de presque chaque date de test. On retire aussi les 2 derniers jours de 2023 de l'entraînement :
leurs cibles à +48 h tombent déjà en 2024.

In [ ]:
tab = pd.read_parquet(os.path.join(DATA, 'table_12h.parquet'))

PARAMS = ['log_flux', 'V_kms', 'Bz_nT', 'F107_sfu', 'N_cm3', 'P_nPa', 'E_mVm', 'B_nT', 'Kp', 'Dst_nT', 'AE_nT']
HORIZONS = [1, 2, 3, 4]                 # en pas de 12 h : +12, +24, +36, +48 h
TEST_START = pd.Timestamp('2024-01-01')

# Entrées X(t) : chaque paramètre à t, t-12 h et t-24 h (shift(k) = valeur k pas de 12 h plus tôt)
X = pd.DataFrame({f'{c}_t-{12 * k}h': tab[c].shift(k) for c in PARAMS for k in (0, 1, 2)})
# Cibles Y(t) : log10 du flux 12, 24, 36, 48 h plus tard (shift(-h) = valeur h pas plus tard)
Y = pd.DataFrame({f'+{12 * h}h': tab['log_flux'].shift(-h) for h in HORIZONS})

# On ne garde que les instants où toutes les entrées et toutes les cibles existent
ok = X.notna().all(axis=1) & Y.notna().all(axis=1)
train = ok & (X.index < TEST_START - pd.Timedelta(hours=12 * max(HORIZONS)))   # cibles avant 2024
test = ok & (X.index >= TEST_START)
X_train, Y_train = X[train], Y[train]
X_test, Y_test = X[test], Y[test]

print(f'{X.shape[1]} entrées : {len(PARAMS)} paramètres x 3 instants (t, t-12 h, t-24 h)')
print(f'Entraînement : {len(X_train)} fenêtres ({X_train.index.min():%Y-%m-%d} -> {X_train.index.max():%Y-%m-%d})')
print(f'Test         : {len(X_test)} fenêtres ({X_test.index.min():%Y-%m-%d} -> {X_test.index.max():%Y-%m-%d})')
print(f'Fenêtres écartées (valeur manquante ou cible à cheval sur 2024) : {(~(train | test)).sum()}')

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(tab.index, tab['log_flux'], color='0.6', lw=0.6)
ax.plot(X_train.index, tab.loc[X_train.index, 'log_flux'], '.', ms=1.5, color='C0', label='entraînement')
ax.plot(X_test.index, tab.loc[X_test.index, 'log_flux'], '.', ms=1.5, color='C1', label='test')
ax.set_ylabel('log10 flux (12 h)')
ax.set_title(f'Découpage dans le temps : entraînement 2020–2023, test 2024')
ax.legend(markerscale=6)
savefig(fig, '8_decoupage.png')
plt.show()